# 🚀 S6E4 - Fast Ensemble (XGB + LGBM)
### Optimized for speed without losing accuracy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print('='*60)
print('S6E4 - FAST ENSEMBLE')
print('='*60)

In [ ]:
# Load data
print('\n[1/5] Loading data...')
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

print(f'  Train: {train.shape}, Test: {test.shape}')

In [ ]:
# Feature Engineering
print('\n[2/5] Feature engineering...')

train['is_train'] = True
test['is_train'] = False
test['Irrigation_Need'] = 'Low'

df = pd.concat([train, test], axis=0, ignore_index=True)

# Encode categoricals
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Interaction features
df['Soil_pH_Soil_Moisture'] = df['Soil_pH'] * df['Soil_Moisture']
df['Temperature_Humidity'] = df['Temperature_C'] * df['Humidity']
df['Rainfall_Soil_Moisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']

# Ratio features
df['Organic_to_Soil'] = df['Organic_Carbon'] / (df['Soil_Moisture'] + 1e-6)
df['EC_to_Soil_pH'] = df['Electrical_Conductivity'] / (df['Soil_pH'] + 1e-6)
df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Prev_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-6)

# Domain features
df['Water_Stress'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + df['Soil_Moisture'] + 1e-6)
df['Irrigation_Efficiency'] = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + 1e-6)
df['Crop_Water_Demand'] = df['Sunlight_Hours'] * df['Temperature_C'] / (df['Humidity'] + 1e-6)

# Target encoding
for group_col in ['Crop_Type', 'Soil_Type', 'Region', 'Season']:
    train_mask = df['is_train'] == True
    group_means = df[train_mask].groupby(group_col)['Irrigation_Need'].apply(
        lambda x: (x == 'High').mean() * 2 + (x == 'Medium').mean()
    )
    df[f'{group_col}_target_mean'] = df[group_col].map(group_means)
    group_counts = df[train_mask].groupby(group_col).size()
    df[f'{group_col}_count'] = df[group_col].map(group_counts)

# Split back
train_final = df.iloc[:len(train)].copy()
test_final = df.iloc[len(train):].copy()
train_final['Irrigation_Need'] = train['Irrigation_Need']

train_final.drop(['is_train', 'id'], axis=1, inplace=True)
test_final.drop(['is_train'], axis=1, inplace=True)
if 'Irrigation_Need' in test_final.columns:
    test_final.drop(['Irrigation_Need'], axis=1, inplace=True)
if 'id' in test_final.columns:
    test_final.drop(['id'], axis=1, inplace=True)

print(f'  Features: {train_final.shape[1] - 1}')

In [ ]:
# Prepare data
print('\n[3/5] Preparing...')

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(train_final['Irrigation_Need'])
X = train_final.drop('Irrigation_Need', axis=1)
X_test = test_final.copy()

print(f'  Classes: {target_encoder.classes_}')
print(f'  Features: {X.shape[1]}')

In [ ]:
# Train models with CV
print('\n[4/5] Training...')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
test_xgb = np.zeros((len(X_test), 3))
test_lgb = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n  Fold {fold+1}/{N_SPLITS}')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # XGBoost (fast with hist)
    xgb_model = xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, eval_metric='mlogloss',
        tree_method='hist', verbosity=0
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_model.predict(X_val)
    print(f'    XGB: {accuracy_score(y_val, oof_xgb[val_idx]):.6f}')
    
    # LightGBM (very fast)
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=-1
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                            lgb.log_evaluation(0)])
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    print(f'    LGBM: {accuracy_score(y_val, oof_lgb[val_idx]):.6f}')
    
    # Test predictions
    test_xgb += xgb_model.predict_proba(X_test) / N_SPLITS
    test_lgb += lgb_model.predict_proba(X_test) / N_SPLITS

# OOF scores
print(f'\n  OOF Accuracies:')
print(f'    XGB:  {accuracy_score(y, oof_xgb):.6f}')
print(f'    LGBM: {accuracy_score(y, oof_lgb):.6f}')

In [ ]:
# Create submission
print('\n[5/5] Creating submission...')

# Ensemble
test_ensemble = (test_xgb + test_lgb) / 2
test_pred = np.argmax(test_ensemble, axis=1)
test_labels = target_encoder.inverse_transform(test_pred)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': test_labels
})

print(f'\n  Predictions:\n{submission["Irrigation_Need"].value_counts()}')

submission.to_csv('submission.csv', index=False)
print('\n  ✓ submission.csv saved!')
print('\n' + '='*60)
print('DONE!')